In [37]:
!pip install face_recognition opencv-python pandas

In [38]:
import os
import cv2
import numpy as np
import pandas as pd
import face_recognition
from datetime import datetime
from base64 import b64decode
from IPython.display import display, Javascript
from google.colab import output
import uuid
import shutil
import pytz  # For timezone handling

# ===========================
# Take Photo (Colab)
# ===========================
def take_photo(filename="photo.jpg", quality=0.8):
    js = Javascript("""
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const btn = document.createElement('button');
      btn.textContent = '📸 Capture';
      btn.style.fontSize = '20px';
      btn.style.margin = '10px';
      div.appendChild(btn);

      const video = document.createElement('video');
      video.style.display = 'block';
      video.style.border = '2px solid black';
      div.appendChild(video);

      document.body.appendChild(div);

      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      video.srcObject = stream;
      await video.play();

      await new Promise(resolve => btn.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);

      stream.getTracks().forEach(track => track.stop());
      div.remove();

      return canvas.toDataURL('image/jpeg', quality);
    }
    """)
    display(js)
    data = output.eval_js("takePhoto(0.8)")
    binary = b64decode(data.split(",")[1])
    with open(filename, "wb") as f:
        f.write(binary)
    print("✅ Image captured successfully")

# ===========================
# Load Registered Faces
# ===========================
def load_faces():
    encodings = []
    ids = []
    names = []
    if not os.path.exists("registered_faces"):
        os.makedirs("registered_faces")

    for person in os.listdir("registered_faces"):
        path = f"registered_faces/{person}"
        for file in os.listdir(path):
            if file.endswith(".jpg"):
                img_path = f"{path}/{file}"
                img = face_recognition.load_image_file(img_path)
                enc = face_recognition.face_encodings(img)
                if enc:
                    encodings.append(enc[0])
                    ids.append(file.split(".")[0])
                    names.append(person)
    return encodings, ids, names

# ===========================
# Register Employee
# ===========================
def register_employee():
    name = input("Enter Employee Name: ").strip()
    if not name:
        print("❌ Name cannot be empty")
        return

    known_encodings, known_ids, known_names = load_faces()
    os.makedirs(f"registered_faces/{name}", exist_ok=True)

    print("📸 Click CAPTURE button to register face")
    emp_id = uuid.uuid4().hex[:8]
    img_path = f"registered_faces/{name}/{emp_id}.jpg"
    take_photo(img_path)

    img = face_recognition.load_image_file(img_path)
    locations = face_recognition.face_locations(img, model="hog")
    if len(locations) == 0:
        print("❌ No face detected. Try again.")
        os.remove(img_path)
        return

    enc = face_recognition.face_encodings(img, locations)[0]

    # 🔹 Show embeddings
    print("Face embedding vector (128-d):")
    print(enc)
    print("Length of vector:", len(enc))

    if known_encodings:
        matches = face_recognition.compare_faces(known_encodings, enc, tolerance=0.5)
        if True in matches:
            print("❌ This face is already registered with another ID")
            os.remove(img_path)
            return

    if not os.path.exists("attendance.csv"):
        df = pd.DataFrame(columns=["Employee_ID","Name","Date","Time"])
        df.to_csv("attendance.csv", index=False)

    print(f"✅ {name} registered successfully with ID: {emp_id}")

# ===========================
# Mark Attendance with Confidence and Embeddings
# ===========================
def mark_attendance_webcam():
    known_encodings, known_ids, known_names = load_faces()
    if not known_names:
        print("⚠️ No registered employees")
        return

    print("📸 Click CAPTURE button to mark attendance")
    take_photo("temp.jpg")

    img = face_recognition.load_image_file("temp.jpg")
    locations = face_recognition.face_locations(img, model="hog")
    if len(locations) == 0:
        print("❌ No face detected")
        return

    encodings = face_recognition.face_encodings(img, locations)
    df = pd.read_csv("attendance.csv")

    # Set timezone to Pakistan (UTC+5)
    tz = pytz.timezone("Asia/Karachi")
    now = datetime.now(tz)
    date_today = now.strftime("%Y-%m-%d")
    time_now = now.strftime("%H:%M:%S")

    for enc in encodings:
        distances = face_recognition.face_distance(known_encodings, enc)
        matches = face_recognition.compare_faces(known_encodings, enc)

        # 🔹 Show embeddings for detected face
        print("Detected face embedding vector (128-d):")
        print(enc)
        print("Length of vector:", len(enc))

        if True in matches:
            best = np.argmin(distances)
            emp_id = known_ids[best]
            name = known_names[best]
            confidence = (1 - distances[best]) * 100  # Convert distance to confidence %
            confidence = round(confidence, 2)

            # Only one attendance per day per employee
            if not ((df["Employee_ID"] == emp_id) & (df["Date"] == date_today)).any():
                df.loc[len(df)] = [emp_id, name, date_today, time_now]
                df.to_csv("attendance.csv", index=False)
                print(f"✅ Attendance marked for {name} | Employee_ID: {emp_id} | Confidence: {confidence}% | Time: {time_now}")
            else:
                print(f"ℹ️ {name} already marked today | Employee_ID: {emp_id} | Confidence: {confidence}%")
        else:
            print("⚠️ Face not registered")

# ===========================
# Show Attendance
# ===========================
def show_attendance():
    if os.path.exists("attendance.csv"):
        df = pd.read_csv("attendance.csv")
        df = df.sort_values(by=["Date","Time"]).reset_index(drop=True)
        display(df)
    else:
        print("No attendance data found")

# ===========================
# Delete Employee
# ===========================
def delete_employee(emp_id):
    # Remove from attendance
    if os.path.exists("attendance.csv"):
        df = pd.read_csv("attendance.csv")
        df = df[df["Employee_ID"] != emp_id]
        df.to_csv("attendance.csv", index=False)
        print(f"✅ Employee {emp_id} removed from attendance records")

    # Remove face image
    removed = False
    for person in os.listdir("registered_faces"):
        if emp_id + ".jpg" in os.listdir(f"registered_faces/{person}"):
            os.remove(f"registered_faces/{person}/{emp_id}.jpg")
            print(f"✅ Employee face {emp_id} removed from system")
            removed = True
            break
    if not removed:
        print("⚠️ Employee ID not found")

# ===========================
# Delete All Data
# ===========================
def delete_all_data():
    if os.path.exists("attendance.csv"):
        os.remove("attendance.csv")
        print("✅ Attendance CSV cleared")
    if os.path.exists("registered_faces"):
        shutil.rmtree("registered_faces")
        print("✅ All registered face images removed")


In [52]:
register_employee()


Enter Employee Name: qasim
📸 Click CAPTURE button to register face


<IPython.core.display.Javascript object>

✅ Image captured successfully
Face embedding vector (128-d):
[-0.15601793 -0.00601995  0.07393111 -0.05846278 -0.03778732 -0.07146375
  0.01880127 -0.08141885  0.22402701 -0.16657768  0.22296403 -0.04622635
 -0.17528409 -0.15077785  0.02473626  0.11071196 -0.13920745 -0.19165081
 -0.07038205 -0.11799599  0.07938605 -0.03421415 -0.01994108  0.05754041
 -0.24754496 -0.33321267 -0.10734273 -0.09124782  0.00675949 -0.04849711
 -0.05106696  0.06478301 -0.20641883 -0.06719723 -0.02094753  0.15450998
  0.01642077 -0.01576359  0.12698102 -0.01642415 -0.15394232 -0.0135019
  0.03605727  0.21797568  0.16740608  0.04036497  0.05407607 -0.02607392
  0.15306666 -0.27364126  0.027036    0.12164753  0.14258587  0.06008724
  0.11966142 -0.17247146  0.02579426  0.13160758 -0.18114777  0.11569905
 -0.01611258 -0.11427379 -0.01843222 -0.01894839  0.18450662  0.10286631
 -0.12241098 -0.09564502  0.16350569 -0.17523803 -0.00227538  0.07893354
 -0.10098856 -0.20957869 -0.33350089  0.04553884  0.48371542  0.

In [58]:
mark_attendance_webcam()


⚠️ No registered employees


In [57]:
show_attendance()

,Employee_ID,Name,Date,Time


In [56]:
delete_employee("34ab5966")

✅ Employee 34ab5966 removed from attendance records
✅ Employee face 34ab5966 removed from system


In [46]:
delete_all_data()

✅ Attendance CSV cleared
✅ All registered face images removed
